# Model Evaluation & Inference
### Select and load a fine-tuned Qwen3 spam classification model to run inference tests.

In [ ]:
import os
import datetime
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftConfig, PeftModel

: 

In [2]:
def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

DEVICE = find_device()
print(f"Using device: {DEVICE.upper()}")

# Global variables to hold our loaded artifacts
model = None
tokenizer = None
MAX_SEQ_LENGTH = 448


Using device: MPS


## Select and Load Model

In [3]:

base_dir = "./results"
prefix = "qwen3_0.6b_spam_saved_weights_"

def get_available_models(base_dir, prefix):
    if not os.path.exists(base_dir):
        return []

    valid_models = []
    for folder_name in os.listdir(base_dir):
        if folder_name.startswith(prefix):
            full_path = os.path.join(base_dir, folder_name)
            if os.path.isdir(full_path):
                timestamp_str = folder_name.replace(prefix, "")
                try:
                    # Convert string back to datetime to ensure chronological sorting
                    folder_dt = datetime.datetime.strptime(timestamp_str, "%H%M%d%m%Y")
                    valid_models.append((folder_dt, folder_name, full_path))
                except ValueError:
                    continue
                    
    # Sort descending (newest first)
    valid_models.sort(key=lambda x: x[0], reverse=True)
    return valid_models

models_list = get_available_models(base_dir, prefix)

if not models_list:
    print(f"❌ No models found in {base_dir} starting with {prefix}")
else:
    # Create dropdown options: "YYYY-MM-DD HH:MM (folder_name)" -> path
    options = {f"{dt.strftime('%Y-%m-%d %H:%M')} ({name})": path for dt, name, path in models_list}
    
    dropdown = widgets.Dropdown(
        options=options,
        description='Model:',
        disabled=False,
        layout={'width': 'max-content'}
    )
    
    load_button = widgets.Button(
        description='Load Selected Model',
        button_style='success',
        icon='check'
    )
    
    output = widgets.Output()

    def on_load_button_clicked(b):
        global model, tokenizer
        with output:
            clear_output()
            selected_path = dropdown.value
            print(f"⏳ Loading adapter from: {selected_path}...")
            
            try:
                # 1. Read PEFT config to know which base model to load
                peft_config = PeftConfig.from_pretrained(selected_path)
                
                # 2. Load the base model (in your training script, it was Qwen/Qwen3-0.6B)
                base_model = AutoModelForSequenceClassification.from_pretrained(
                    peft_config.base_model_name_or_path,
                    num_labels=2,
                    # Optional: match your training dtypes if needed, e.g., torch_dtype=torch.bfloat16
                )
                
                # 3. Apply the LoRA adapter
                model = PeftModel.from_pretrained(base_model, selected_path)
                model.to(DEVICE)
                model.eval()
                
                # 4. Load the tokenizer
                tokenizer = AutoTokenizer.from_pretrained(selected_path)
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token
                tokenizer.padding_side = "right" 
                
                model.config.pad_token_id = tokenizer.pad_token_id                
                print("✅ Model and tokenizer successfully loaded and moved to", DEVICE.upper())
            except Exception as e:
                print(f"❌ Failed to load model: {e}")

    load_button.on_click(on_load_button_clicked)
    
    # Display the UI
    display(widgets.HBox([dropdown, load_button]), output)


Output()

## Inference Function

In [4]:
def _label_to_str(pred: int) -> str:
    return "spam" if pred == 1 else "valid"

def run_mail_classification(email_text: str) -> str:
    """Classify a raw email string as 'spam' or 'valid' using the loaded model."""
    if model is None or tokenizer is None:
        raise RuntimeError("Model is not loaded! Please select and load a model using the widget above.")
        
    email_text = email_text.strip()
    print("-" * 40)
    print(f"TEXT:\n{email_text}")
    print("-" * 40)
    
    inputs = tokenizer(
        email_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    pred = torch.argmax(outputs.logits, dim=-1).item()
    result = _label_to_str(pred)
    
    print(f"PREDICTION: ---> {result.upper()} <---")
    return result


## Tests

In [ ]:
run_mail_classification("""\
Subject: E-mail details of the client.
Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards, John
""")

In [ ]:
run_mail_classification("""\
Subject: Free iPhone.
Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
""")

In [ ]:
run_mail_classification("""
Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the ‘Cards’ tab in the app to access your new card details and start making payments online.
""")

In [ ]:
run_mail_classification("""\
Subject: Obsługa języka polskiego.
Czy język poslski jest obsługiwany przez nasz interpreter spamu? Ostatnio dostaję duzo wiadomości, które trafiają do spamu a nie powinny.
Pozdrawiam, Wojciech
""")

In [ ]:
run_mail_classification("""
Subject: Todays meeting.
Hi Josh, I just wanted to make sure that the meeting today is still on since you were sick yesterday. The matter of finding porn on your laptop is really important and may lead to your termination.
""")

## Evaluate on Original Test Split

In [ ]:
from datasets import load_dataset, ClassLabel
import numpy as np
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from tqdm.auto import tqdm

# Original splitting parameters
DATASET_PATH = "../dataset/combined_datasets/spam_2007_2008/dataset.parquet"
SEED = 67
VALIDATION_SPLIT = 0.05
TEST_SPLIT = 0.05
HOLDOUT_SPLIT = VALIDATION_SPLIT + TEST_SPLIT

if model is None or tokenizer is None:
    raise RuntimeError("Model is not loaded! Please select and load a model using the widget above.")

# 1. Load and cast
raw_dataset = load_dataset("parquet", data_files=DATASET_PATH, split="train")
raw_dataset = raw_dataset.cast_column("label", ClassLabel(names=["valid", "spam"]))

# 2. Recreate splits
holdout = raw_dataset.train_test_split(
    test_size=HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)
valid_test = holdout["test"].train_test_split(
    test_size=TEST_SPLIT / HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)
test_dataset = valid_test["test"]

# 3. Apply text transform
def dataset_transform(sample):
    subject = (sample["subject"] or "").strip()
    body = (sample["body"] or "").strip()
    parts = []
    if subject:
        parts.append(f"Subject: {subject}")
    if body:
        parts.append(body)
    sample["text"] = "\n\n".join(parts).strip()
    return sample

test_dataset = test_dataset.map(dataset_transform, desc="Building email texts")
print(f"✅ Reconstructed test split with {len(test_dataset)} samples.")

# %% [markdown]
# ### Run Batch Evaluation

# %%
# 1. Tokenize the test set
def tokenize(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    encoded["length"] = [len(ids) for ids in encoded["input_ids"]]
    return encoded

tokenized_test = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["subject", "body", "source", "text"], # Remove strings, keep label
    desc="Tokenizing test set"
)

tokenized_test = tokenized_test.rename_column("label", "labels")

# Filter out empty sequences exactly as done in training
tokenized_test = tokenized_test.filter(lambda x: x["length"] > 0, desc="Filtering empty sequences")

# 2. Setup DataLoader
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
test_dataloader = DataLoader(tokenized_test, batch_size=8, collate_fn=data_collator)

# 3. Evaluation Loop
model.eval()
all_preds = []
all_labels = []

print("🚀 Running evaluation on test split...")
for batch in tqdm(test_dataloader):
    # Move batch to device (mps/cuda/cpu)
    batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "length"}
    
    with torch.no_grad():
        outputs = model(**batch)
    
    preds = torch.argmax(outputs.logits, dim=-1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 4. Compute Metrics
accuracy = float((all_preds == all_labels).mean())
tp = int(((all_preds == 1) & (all_labels == 1)).sum())
fp = int(((all_preds == 1) & (all_labels == 0)).sum())
fn = int(((all_preds == 0) & (all_labels == 1)).sum())

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-12)

print("\n📊 Test Split Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Mistakes:  {sum(all_preds != all_labels)} / {len(all_labels)}")

In [ ]:
incorrect_indices = np.where(all_preds == 1)[0]

print(f"Total mistakes: {len(incorrect_indices)}\n")
print("=" * 80)

for idx in incorrect_indices:
    # Convert numeric labels back to strings
    actual_label = _label_to_str(all_labels[idx])
    pred_label = _label_to_str(all_preds[idx])
    
    # Extract the exact text provided to the model
    raw_text = test_dataset[int(idx)]["text"]
    
    print(f"Test Index: {idx}")
    print(f"Actual: {actual_label.upper()} | Predicted: {pred_label.upper()}")
    print("-" * 80)
    print(raw_text)
    print("=" * 80)
    print("\n")